In [6]:
import os, json
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
from pygms import TargetIntensityMeasure, GroundMotionSelection

In [7]:
# load configuration file
with open('./example1.json','r') as f:
    job_config = json.load(f)

In [8]:
# create a target intensity measure
tgt_config = job_config.get('TargetIntensityMeasure')
my_tgt = TargetIntensityMeasure(tgt_config)
my_tgt.run_im_calculator()

USGS_GMM: GMM configured


In [ ]:
# ground motion selection
gms_config = job_config.get('GroundMotionSelection')
gms_config.update({'TargetType':tgt_config.get('TargetType')})
my_gms = GroundMotionSelection(gms_config)
my_gms.generate_pseudo_im(my_tgt.im_target)
my_gms.initial_scanning(my_tgt.ims,my_tgt.cond_imv,my_tgt.cond_im_idx)
my_gms.optimize_selection(my_tgt.ims)
my_gms.export_selection(my_tgt.ims)

GroundMotionSelection.optimize_selection: current total error: 0.04723751023601441.
GroundMotionSelection.optimize_selection: 0.5%.
GroundMotionSelection.optimize_selection: current total error: 0.04500445898257971.
GroundMotionSelection.optimize_selection: 1.0%.
GroundMotionSelection.optimize_selection: current total error: 0.04401210744995139.
GroundMotionSelection.optimize_selection: 1.5%.
GroundMotionSelection.optimize_selection: current total error: 0.04401210744995139.
GroundMotionSelection.optimize_selection: 2.0%.
GroundMotionSelection.optimize_selection: current total error: 0.04368309565525312.
GroundMotionSelection.optimize_selection: 2.5%.
GroundMotionSelection.optimize_selection: current total error: 0.04368309565525312.
GroundMotionSelection.optimize_selection: 3.0%.
GroundMotionSelection.optimize_selection: current total error: 0.04368309565525312.
GroundMotionSelection.optimize_selection: 3.5%.
GroundMotionSelection.optimize_selection: current total error: 0.04368309565

In [ ]:
# plot
plt.loglog(my_gms.selection_periods, np.exp(my_gms.gmdb_imv[np.ix_(my_gms.selected_id,my_gms.ims_idx[0:len(my_gms.selection_periods)])]).T,color='k',linewidth=0.5)
plt.loglog(my_gms.selection_periods, np.exp(np.mean(my_gms.gmdb_imv[np.ix_(my_gms.selected_id,my_gms.ims_idx[0:len(my_gms.selection_periods)])],axis=0)).T,color='r',linestyle='--')
plt.loglog(my_gms.selection_periods, np.exp(my_gms.tgt_mean[0:len(my_gms.selection_periods)]),color='b',linestyle='-')
# Add labels and title
plt.xlabel('T (sec)')
plt.ylabel('Sa (g)')
plt.xlim([0.1,10])
plt.grid()
# Show plot
plt.show()

if my_gms.tgt_type in ['CSD','CS']:
    plt.plot(my_gms.selection_periods, np.std(my_gms.gmdb_imv[np.ix_(my_gms.selected_id,my_gms.ims_idx[0:len(my_gms.selection_periods)])],axis=0),color='r',linestyle='--')
    plt.plot(my_gms.selection_periods, np.sqrt(np.diag(my_gms.tgt_cov))[0:len(my_gms.selection_periods)],color='b',linestyle='-')
    # Add labels and title
    plt.xlabel('T (sec)')
    plt.ylabel('std(lnSa)')
    plt.xlim([0.1,10])
    plt.grid()
    # Show plot
    plt.show()

if my_gms.tgt_type in ['CSD']:
    x = np.sort(np.exp(my_gms.gmdb_imv[my_gms.selected_id,-1]))
    y = np.arange(1,my_gms.num_rec+1)/my_gms.num_rec
    plt.plot(x, y,color='k',linewidth=1)
    x = np.linspace(0.5,50,50)
    y = norm.cdf(np.log(x), loc=my_gms.tgt_mean[-1], scale=np.sqrt(np.diag(my_gms.tgt_cov))[-1])
    plt.plot(x, y,color='b',linewidth=1)
    # Add labels and title
    plt.xlabel('Ds5-75 (sec)')
    plt.ylabel('CDF')
    plt.grid()
    # Show plot
    plt.show()